In [ ]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

from tqdm import tqdm

from loka.data import load_breakthrough_states
from loka.models import BreakthroughCNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
filename = "/home/axelbjarkar/RU/T-404-LOKA-code/data/e.txt"
X,Y = load_breakthrough_states(filename)
X = X.reshape(-1, 2, 5, 5)

In [ ]:
start = 0.01
end = 0.9
steps = 50
batch_size = 512

splits = np.linspace(start, end, num=steps, endpoint=True)
mse_values = []
prc_values = []

epochs = 10
pbar = tqdm(total=steps * epochs, desc='training')

for seed, ratio in enumerate(splits):
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=1-ratio, random_state=seed)

    torch.manual_seed(SEED)  # same init weights each time
    model = BreakthroughCNN().to(device)

    # training setup
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()
    
    # convert data to tensors)
    X_train_t = torch.FloatTensor(X_train).to(device)
    Y_train_t = torch.FloatTensor(Y_train).to(device)
    X_t = torch.FloatTensor(X).to(device)
    Y_t = torch.FloatTensor(Y).to(device)
    
    # training loop
    for epoch in range(epochs):
        perm = torch.randperm(len(X_train_t), device=device)  # new shuffle each epoch
        
        for i in range(0, len(X_train_t), batch_size):
            idx = perm[i:i + batch_size]
            xb, yb = X_train_t[idx], Y_train_t[idx]
    
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
        pbar.update(1)

    pbar.set_postfix(model=f'{seed+1}/{steps}', ratio=f'{ratio:.2f}', loss=f'{loss.item():.4f}')
    
    # evaluate
    with torch.no_grad():
        test_preds = model(X_t)
        mse = loss_fn(test_preds, Y_t).item()
        mse_values.append(mse)
        prc_values.append(ratio)

pbar.close()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))

output_dir = '/home/axelbjarkar/RU/T-404-LOKA-code/experiments/M1/results'
output_file = f'{output_dir}/random_subset_larger.png'

ax.plot(np.array(prc_values) * 100, mse_values, 'o-', color='#2196F3', linewidth=2, markersize=8, markerfacecolor='white', markeredgewidth=2)

ax.set_xlabel(r"$\frac{|S|}{|E|}$ (%)", fontsize=20)
ax.set_ylabel("MSE", fontsize=20)
ax.set_title("Model Performance vs Training Subset Size", fontsize=22)

ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(output_file)
plt.show()